# Prioritizing Content Refresh Opportunities with Search Performance Signals

**Lane:** Refresh / Content Opportunity Scoring

This capstone mirrors the deployed research paper. It rebuilds the March 2026 public-safe frame, compares the frozen Week-4 baseline with readable learned rankings under a client-grouped holdout, audits leakage, and produces a ranked human-review playbook. Headline metrics are computed live from the gated FlyRank warehouse — never replace them with starter/example numbers.


## Abstract

Content teams have limited review capacity, so the practical question is not “which page is bad?” but **which pages should be reviewed first?** This study uses the FlyRank ML Internship warehouse to build a public-safe March 2026 feature frame from pre-decision search signals. A transparent CTR-versus-position baseline is compared with Logistic Regression, a small Decision Tree, and Random Forest using the same client-grouped holdout and Precision@K metrics. The live results table below determines whether a learned ranking improves the top of the review queue over the frozen rule. The output is directional decision support for human reviewers, not proof that editing a recommended page will cause recovery.


## 1. Introduction / Problem statement

A content reviewer can inspect only a small fraction of a large portfolio at one time. A useful system therefore needs to rank review candidates, provide a reason a human can inspect, and keep uncertainty visible.

> **Research question:** Which already-visible pages should a content team review first for refresh or optimization, using only search-performance signals available before the decision moment?

The action is human review. A high score means “inspect earlier,” not “automatically rewrite.”


In [ ]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn matplotlib
import os,sys,json
from pathlib import Path
import pandas as pd,numpy as np
from google.colab import userdata
from huggingface_hub import whoami
REPO_DIR=Path('/content/Internship')
if not REPO_DIR.exists():
    !git clone https://github.com/imalik-7/Internship.git /content/Internship
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR))
from work.lib.capstone_pipeline import FEATURES,LABEL,CONTEXT,connect_warehouse,build_analysis_frame,grouped_split,train_compare,best_model_name,feature_importance,ensure_output_dir
HF_TOKEN=userdata.get('HF_TOKEN'); print('HF account:',whoami(token=HF_TOKEN)['name'])
con=connect_warehouse(HF_TOKEN); analysis_df=build_analysis_frame(con)
print('Unit: one pseudonymized client-content item')
print('Rows:',len(analysis_df),'| positive proxy rate:',round(analysis_df[LABEL].mean(),3))


## 2. Data

**Release:** FlyRank ML Internship warehouse on Hugging Face. **Primary table:** `fact_content_daily_performance`. **Development slice:** March 2026, a mid-panel month rather than the final-month sample. **Feature window:** March 1–15. **Outcome window:** March 16–30. **Population:** pseudonymized client-content items with at least 100 feature-window impressions.

Excluded from model features: outcome-window impressions, the decline proxy itself, existing FlyRank product flags/scores, raw client names, domains, URLs, titles, private queries, and credentials. One model row represents one pseudonymized content item within a client.


In [ ]:
print('Five honest features:',FEATURES)
print('Feature window: 2026-03-01 to 2026-03-15')
print('Outcome window: 2026-03-16 to 2026-03-30')
print('Rows:',len(analysis_df))
print('Unique client-content pairs:',analysis_df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0])


## 3. Methodology

### Features
Exactly five pre-decision aggregates: feature-window impressions, clicks, CTR, impression-weighted average position, and active impression days.

### Label / proxy
`is_declining_proxy = 1` when March 16–30 impressions are more than 20% below March 1–15 impressions. This is a short-window decline proxy, not a causal refresh label.

### Baseline
The frozen Week-4 rule prioritizes pages with at least 500 impressions, position 1–20, and CTR at least 20% below the median CTR for the same position tier. Its score is visibility × CTR-gap severity.

### Learned methods and validation
Logistic Regression, a small Decision Tree, and Random Forest use probabilities as ranking scores. A fixed-seed client-grouped 75/25 holdout keeps clients from crossing train/test. Baseline thresholds are learned from training data only. Metrics are Precision@10, @20, @50 and Average Precision.

### Leakage checks
All features occur before the outcome window. No label sibling, outcome measurement, or existing product flag is in `FEATURES`.


In [ ]:
train_df,test_df=grouped_split(analysis_df,test_size=0.25,random_state=42)
comparison,fitted_models,test_scores,baseline_test=train_compare(train_df,test_df,random_state=42)
BEST_MODEL=best_model_name(comparison); BASE_RATE=float(test_df[LABEL].mean())
print('Train rows:',len(train_df),'Test rows:',len(test_df))
print('Client overlap:',len(set(train_df.client_hash_id)&set(test_df.client_hash_id)))
print('Held-out base rate:',round(BASE_RATE,3)); print('Best learned model:',BEST_MODEL)


## 4. Results — model vs baseline

The table below is the headline comparison: every method uses the same held-out rows. The base rate stays visible so Precision@K is read in context. A learned model only earns its place where it beats the transparent baseline; mixed results across K are reported rather than hidden.


In [ ]:
print('Held-out base rate:',round(BASE_RATE,3)); display(comparison.round(3))
importance=feature_importance(fitted_models[BEST_MODEL]); display(importance)
import matplotlib.pyplot as plt
OUT=ensure_output_dir(REPO_DIR)
plot_df=comparison.set_index('method')[['precision_at_10','precision_at_20','precision_at_50']]
ax=plot_df.plot(kind='bar',figsize=(10,5)); ax.set_ylabel('Precision'); ax.set_xlabel(''); ax.set_title('Baseline vs learned rankings — client-grouped holdout'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.savefig(OUT/'capstone_precision_at_k.png',dpi=160); plt.show()
ax=importance.sort_values('importance').plot(x='feature',y='importance',kind='barh',legend=False,figsize=(8,4)); ax.set_xlabel('Importance / coefficient magnitude'); ax.set_ylabel(''); ax.set_title('What the best learned model leans on'); plt.tight_layout(); plt.savefig(OUT/'capstone_feature_importance.png',dpi=160); plt.show()


## 5. Limitations & honest framing

This is an observational, short-window ranking experiment. The label is a later impression-decline **proxy**, not an observed causal benefit from refreshing content. The analysis uses one mid-panel month and five search-only features; it does not fully model seasonality, query mix, SERP changes, content quality, or business priorities. Client grouping reduces one source of memorization, but a future multi-month time split would better mimic deployment.

The results support **directional prioritization for human review**. They do not prove Google's ranking algorithm, do not establish that a refresh causes recovery, and should not trigger automatic deletion, rewriting, or publishing.


In [ ]:
FORBIDDEN={'impressions_outcome15',LABEL,'baseline_action_score','reason_code','action_label','leaky_outcome_ratio'}
print('Leakage intersection:',set(FEATURES)&FORBIDDEN)
assert not(set(FEATURES)&FORBIDDEN)
print('PASS: final feature list is pre-decision and excludes labels/product outputs.')


## 6. Ranked recommendations

The recommendation layer translates model scores into a queue a reviewer can use. Reason codes describe observable patterns; they are not causal diagnoses. Typical actions are: review title/meta and intent for high-visibility low-CTR pages; review content/query fit for striking-distance pages; monitor when evidence is thin; protect strong search capture; otherwise perform a manual content review. Every action requires human context checking before editing.


In [ ]:
queue=test_df[CONTEXT+FEATURES+[LABEL]].copy(); queue['model_score']=test_scores[BEST_MODEL]
ctr_med=float(train_df.ctr_feature15.median()); imp_q75=float(train_df.impressions_feature15.quantile(.75))
def rec(row):
    if row['impressions_feature15']>=imp_q75 and row['ctr_feature15']<ctr_med: return 'high_visibility_low_ctr','review_title_meta_and_intent'
    if 11<=row['avg_position_feature15']<=20: return 'striking_distance','review_content_and_query_fit'
    if row['active_impression_days_feature15']<8: return 'thin_observation_window','monitor_before_editing'
    if row['avg_position_feature15']<=10 and row['ctr_feature15']>=ctr_med: return 'strong_search_capture','protect_and_monitor'
    return 'multi_signal_review','manual_content_review'
rr=queue.apply(rec,axis=1,result_type='expand'); queue['reason_code'],queue['action_label']=rr[0],rr[1]
queue=queue.sort_values('model_score',ascending=False).reset_index(drop=True); queue['rank']=np.arange(1,len(queue)+1)
public_top10=queue[['rank','content_hash_id','model_score','reason_code','action_label']].head(10); display(public_top10)
queue[['rank','client_hash_id','content_hash_id','model_score','reason_code','action_label']+FEATURES].to_csv(OUT/'capstone_action_queue.csv',index=False)
public_top10.to_csv(OUT/'capstone_public_top10.csv',index=False)


## 7. Reproducibility

Repository: **https://github.com/imalik-7/Internship**

Core notebooks: `w01_research_question.ipynb`, `w02_ml_task_framing.ipynb`, `w03_data_contract.ipynb`, `w04_baseline_score.ipynb`, `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`, and `capstone.ipynb` under `work/notebooks/`. Random seed: **42**. The gated warehouse token is read from Colab Secrets and is never committed.


In [ ]:
payload={'title':'Prioritizing Content Refresh Opportunities with Search Performance Signals','lane':'Refresh / Content Opportunity Scoring','analysis_rows':int(len(analysis_df)),'train_rows':int(len(train_df)),'test_rows':int(len(test_df)),'test_base_rate':BASE_RATE,'best_learned_model':BEST_MODEL,'comparison':comparison.to_dict(orient='records'),'features':FEATURES,'random_seed':42}
with open(OUT/'capstone_metrics.json','w') as f: json.dump(payload,f,indent=2)
print('Wrote',OUT/'capstone_metrics.json')


## 8. Acknowledgments & data credit

**Built on the FlyRank ML Internship dataset.** Data credit: [FlyRank](https://flyrank.ai).

Thanks to the FlyRank internship track for the public-safe warehouse release, lane framework, and emphasis on baselines, leakage checks, honest validation, and human-readable actions.


## 9. ML-12 communication cuts

### 5-minute demo outline
1. Problem: limited review capacity. 2. Data contract: March 1–15 features → March 16–30 proxy. 3. Frozen transparent baseline. 4. Same-split model comparison and validation. 5. Reason-coded action queue. 6. Limits: decision support, not causal proof.

### Social-post cut
I built a content-review ranking system on the FlyRank ML Internship warehouse. I froze a transparent CTR/position baseline, trained readable models on the same five pre-decision search signals, and evaluated everything on a client-grouped holdout. The final output is a ranked, reason-coded review queue with explicit leakage checks and limits.

### Employer-facing 3-sentence summary
I built an end-to-end search-intelligence capstone that turns warehouse-scale search data into a ranked content-review queue. I designed the label timeline, leakage-safe features, transparent baseline, client-grouped validation, model comparison, error analysis, and human action layer. The model only earns its place when it beats a simpler rule under the same honest evaluation.


## Self-check

- [x] All nine paper sections are present, including Abstract and Acknowledgments/data credit.
- [x] Question, decision, data windows, exclusions, methodology, baseline, models, grouped split, and leakage checks are explicit.
- [x] Results compare baseline and learned models on the same held-out rows with base rate visible.
- [x] Limitations prohibit causal / Google-algorithm claims.
- [x] Ranked recommendations use pseudonymous identifiers and human review.
- [x] Reproducibility links and seed are present.
- [x] 5-minute demo, social cut, and employer-facing summary are included.
